### Data Description and Preparation

The electricity consumption data is originally provided at a 30-minute resolution for each French region. To combine it with the weather dataset, which is only available at a daily frequency, we aggregate electricity consumption to the daily level.

For each region and each day, all 30-minute consumption values are summed to obtain the total daily electricity demand. The resulting daily dataset is then merged with the daily weather data using the common date and region identifiers.

This ensures both datasets are aligned at the same temporal resolution before descriptive analysis and modeling.

In [2]:
pip install -q cartiflette

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
import plotly.express as px

In [4]:
# ele = pd.read_csv("/home/onyxia/work/Python_Project_2A/data/RTE/data_ele-2020-2024.csv")
# climate = pd.read_csv("/home/onyxia/work/Python_Project_2A/data/weather/Region_France_weather_daily_2020_2024.csv")

df = pd.read_csv("/home/onyxia/work/Python_Project_2A/data/temp_electricity.csv")

In [5]:
# ele

In [6]:
# climate

1. **Does electricity consumption vary systematically with temperature?**  

In [7]:
df

,Datetime,Consommation,Regions,Nature,year,month,day,ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,GWETROOT,...,QV2M,RH2M,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,WD10M,WS10M,WS2M,insee_dep
0,2020-01-01,403787.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,1,5.39,6.80,0.68,...,4.17,96.11,0.81,5.26,-1.36,6.62,314.6,1.44,0.82,83
1,2020-01-02,443531.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,2,4.78,6.45,0.68,...,4.41,92.78,2.44,8.35,-0.89,9.24,200.2,3.64,2.26,83
2,2020-01-03,434626.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,3,3.80,6.38,0.68,...,5.00,93.72,3.94,8.16,1.49,6.67,246.6,3.21,1.91,83
3,2020-01-04,395169.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,4,3.91,6.99,0.68,...,4.76,94.68,3.18,5.62,-0.64,6.26,335.6,4.34,2.77,83
4,2020-01-05,400169.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,5,5.85,7.13,0.67,...,3.96,93.92,0.77,4.06,-1.25,5.31,354.9,3.75,2.43,83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21919,2024-12-27,181429.0,Pays-de-la-Loire,Données consolidées,2024,12,27,2.05,5.86,0.55,...,3.91,94.54,1.20,5.31,-3.47,8.78,115.8,1.72,0.97,52
21920,2024-12-28,171307.0,Pays-de-la-Loire,Données consolidées,2024,12,28,2.10,5.59,0.55,...,4.93,98.06,4.17,5.52,3.20,2.32,188.7,1.12,0.68,52
21921,2024-12-29,169980.0,Pays-de-la-Loire,Données consolidées,2024,12,29,2.32,5.64,0.55,...,4.17,93.09,2.60,4.63,0.94,3.69,72.1,1.22,0.80,52
21922,2024-12-30,185431.0,Pays-de-la-Loire,Données consolidées,2024,12,30,2.34,5.69,0.55,...,3.90,95.37,1.30,2.89,-0.17,3.06,172.1,1.81,1.14,52


In [8]:
df.describe()

,Consommation,year,month,day,ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,GWETROOT,GWETTOP,PRECTOTCORR,PS,QV2M,RH2M,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,WD10M,WS10M,WS2M,insee_dep
count,21924.000000,21924.00000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000
mean,204433.967843,2022.00000,6.521073,15.735085,12.832862,17.736459,0.575099,0.596883,1.994065,98.300271,7.198033,79.303629,12.013618,16.923766,7.537540,9.386226,195.470799,4.279693,2.812103,49.833333
std,94293.008281,1.41502,3.449291,8.802592,7.962586,8.521331,0.129210,0.138613,3.699876,2.955280,2.469397,13.039571,6.930725,7.922475,6.030541,3.566004,100.094157,1.967521,1.393298,25.531065
min,58376.000000,2020.00000,1.000000,1.000000,0.460000,3.730000,0.220000,0.100000,0.000000,87.760000,1.700000,23.680000,-8.040000,-3.690000,-12.570000,0.680000,0.000000,0.620000,0.400000,11.000000
25%,125892.500000,2021.00000,4.000000,8.000000,5.710000,9.320000,0.480000,0.500000,0.030000,97.290000,5.320000,71.980000,6.827500,10.920000,3.100000,6.700000,114.200000,2.820000,1.770000,27.750000
50%,191000.500000,2022.00000,7.000000,16.000000,11.700000,17.800000,0.570000,0.610000,0.360000,99.140000,6.990000,81.770000,11.635000,16.440000,7.570000,9.100000,213.300000,3.890000,2.530000,48.000000
75%,262712.500000,2023.00000,10.000000,23.000000,19.062500,26.110000,0.660000,0.700000,2.350000,100.330000,8.940000,89.630000,17.380000,22.710000,12.170000,11.930000,274.300000,5.340000,3.560000,75.250000
max,624804.000000,2024.00000,12.000000,31.000000,32.360000,32.990000,0.960000,0.940000,53.790000,103.610000,16.360000,100.000000,33.310000,42.330000,24.850000,21.630000,360.000000,15.030000,10.600000,93.000000


In [9]:
df_total = df.groupby("Datetime", as_index=False)["Consommation"].sum()

fig = px.line(df_total, x="Datetime", y="Consommation",
              title="Total Electricity Consumption in France Over Time"
              ,labels={"Consommation": "Consommation (MWh)"})
fig.show()

**Observation :**

In [10]:
from cartiflette import carti_download
import geopandas as gpd
import matplotlib.pyplot as plt
import json

In [11]:
regions_gdf = carti_download(
    values=["France"],
    crs=4326,                             # WGS84
    borders="REGION",                     # we want regions, not communes
    vectorfile_format="geojson",
    simplification=50,                    # simplify geometry (0–50) to make it lighter
    filter_by="FRANCE_ENTIERE",           # whole France; or "FRANCE_ENTIERE_DROM_RAPPROCHES"
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2022
)

In [12]:
regions_gdf

,INSEE_REG,PAYS,LIBELLE_REGION,POPULATION,SOURCE,geometry
0,1,France,Guadeloupe,384239,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-61.62648 16.27275, -61.62658 ..."
1,3,France,Guyane,281678,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-51.6408 4.21282, -51.64144 4...."
2,2,France,Martinique,364508,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-60.93246 14.7306, -60.9326 14..."
3,6,France,Mayotte,256518,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((45.08357 -12.96139, 45.08547 -..."
4,84,France,Auvergne-Rhône-Alpes,8042936,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((6.06386 46.41639, 6.06267 46.4168, 6..."
5,76,France,Occitanie,5933185,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((2.06288 44.97662, 2.06244 44.9..."
6,53,France,Bretagne,3354854,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-3.97907 47.70396, -3.97953 47..."
7,75,France,Nouvelle-Aquitaine,6010289,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-0.31325 42.8494, -0.31227 42...."
8,28,France,Normandie,3325032,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-1.924 49.66512, -1.92439 49.6..."
9,93,France,Provence-Alpes-Côte d'Azur,5081101,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((4.23021 43.46049, 4.2329 43.46..."


In [13]:
regions_gdf.columns

Index(['INSEE_REG', 'PAYS', 'LIBELLE_REGION', 'POPULATION', 'SOURCE',
       'geometry'],
      dtype='object')

In [14]:
# Now i have 2 data which is the my df data and the other one is the regions_gdf. So in order to plot the map of the consumption each region using my data, i need to make sure if the region of my data match with the regions_gdf.

In [15]:
df["Regions"].nunique()
df["Regions"].unique()

array(['Auvergne-Rhône-Alpes', 'Bourgogne-Franche-Comté', 'Bretagne',
       'Centre-Val de Loire', 'Grand-Est', 'Hauts-de-France',
       'Ile-de-France', 'Normandie', 'Nouvelle-Aquitaine', 'Occitanie',
       'PACA', 'Pays-de-la-Loire'], dtype=object)

In [16]:
sorted(df["Regions"].unique())
sorted(regions_gdf["LIBELLE_REGION"].unique())

['Auvergne-Rhône-Alpes',
 'Bourgogne-Franche-Comté',
 'Bretagne',
 'Centre-Val de Loire',
 'Corse',
 'Grand Est',
 'Guadeloupe',
 'Guyane',
 'Hauts-de-France',
 'La Réunion',
 'Martinique',
 'Mayotte',
 'Normandie',
 'Nouvelle-Aquitaine',
 'Occitanie',
 'Pays de la Loire',
 "Provence-Alpes-Côte d'Azur",
 'Île-de-France']

In [17]:
#check if the region match to the regions_gdf data. It does exist but just different spelling so need to correct the spelling in df data

set(df["Regions"].unique()) - set(regions_gdf["LIBELLE_REGION"].unique())

{'Grand-Est', 'Ile-de-France', 'PACA', 'Pays-de-la-Loire'}

In [18]:
#correct spelling

region_mapping = {
    "Grand-Est": "Grand Est",
    "Ile-de-France": "Île-de-France",
    "Pays-de-la-Loire": "Pays de la Loire",
    "PACA": "Provence-Alpes-Côte d'Azur"
}
df["Regions"] = df["Regions"].replace(region_mapping)

df["Regions"].unique()

array(['Auvergne-Rhône-Alpes', 'Bourgogne-Franche-Comté', 'Bretagne',
       'Centre-Val de Loire', 'Grand Est', 'Hauts-de-France',
       'Île-de-France', 'Normandie', 'Nouvelle-Aquitaine', 'Occitanie',
       "Provence-Alpes-Côte d'Azur", 'Pays de la Loire'], dtype=object)

In [19]:
overseas = ["Guadeloupe", "Guyane", "Martinique", "Mayotte", "La Réunion"]

regions_metropole = regions_gdf[
    ~regions_gdf["LIBELLE_REGION"].isin(overseas)
].copy()

regions_geojson = json.loads(regions_metropole.to_json())

In [20]:
df_day_region = (
    df.groupby(["Datetime", "Regions"], as_index=False)["Consommation"]
      .sum()
)

# ensure Datetime is really datetime
df_day_region["Datetime"] = pd.to_datetime(df_day_region["Datetime"])

# string version of the date for the slider
df_day_region["date_str"] = df_day_region["Datetime"].dt.strftime("%Y-%m-%d")

In [22]:
regions_metropole

,INSEE_REG,PAYS,LIBELLE_REGION,POPULATION,SOURCE,geometry
4,84,France,Auvergne-Rhône-Alpes,8042936,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((6.06386 46.41639, 6.06267 46.4168, 6..."
5,76,France,Occitanie,5933185,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((2.06288 44.97662, 2.06244 44.9..."
6,53,France,Bretagne,3354854,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-3.97907 47.70396, -3.97953 47..."
7,75,France,Nouvelle-Aquitaine,6010289,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-0.31325 42.8494, -0.31227 42...."
8,28,France,Normandie,3325032,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-1.924 49.66512, -1.92439 49.6..."
9,93,France,Provence-Alpes-Côte d'Azur,5081101,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((4.23021 43.46049, 4.2329 43.46..."
10,52,France,Pays de la Loire,3806461,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-1.10454 46.31496, -1.10269 46..."
11,44,France,Grand Est,5556219,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((3.41474 48.39019, 3.41443 48.38781, ..."
12,27,France,Bourgogne-Franche-Comté,2805580,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((7.13026 47.50295, 7.13157 47.50344, ..."
13,11,France,Île-de-France,12262544,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((1.5014 48.94103, 1.51182 48.93499, 1..."


In [ ]:
fig = px.choropleth(
    df_day_region,
    geojson=regions_geojson,
    locations="Regions",                
    featureidkey="properties.LIBELLE_REGION",           
    color="Consommation",
    animation_frame="date_str",             
    color_continuous_scale="Viridis",
    labels={"Consommation": "Consumption (MWh)",
            "date_str": "Date"},
    title="Daily Electricity Consumption by Region in France"
)

fig.update_geos(fitbounds="locations", visible=False)
fig.show()